# Quick Satellite GeoTIFF

This notebook downloads a square satellite GeoTIFF around a region of interest and then masks everything outside the original ROI to black.

Workflow:
1. Open this notebook with the `quick_sat_gen` kernel.
2. Draw a polygon on the map.
3. Run the ROI cell to build the square download area.
4. Run the download cell to save `satellite.tif`.

Notes:
- The download area is squared so the imagery comes back as a square tile set.
- The final GeoTIFF keeps only the original ROI; everything outside it is filled with black.
- Step 3 lets you choose a target pixel resolution in meters per pixel.
- The helper converts that target resolution to the nearest web-tile zoom level automatically.

## Setup

Run this cell once per session. It finds the local `utils` package whether you opened the notebook from inside `sat_generation` or from the repo root.

In [1]:
import sys
from pathlib import Path

import ipyleaflet
import leafmap
import rasterio

working_dir = Path.cwd()
if (working_dir / "utils").exists():
    notebook_dir = working_dir
elif (working_dir / "sat_generation" / "utils").exists():
    notebook_dir = working_dir / "sat_generation"
else:
    raise FileNotFoundError("Could not find sat_generation/utils from the current working directory.")

if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

from utils.satellite import bbox_to_polygon, estimate_tms_request, square_bbox, tms_to_geotiff

OUTPUT_IMAGE = notebook_dir / "satellite.tif"
DEFAULT_CENTER = [29.676840, -95.369222]
DEFAULT_MAP_ZOOM = 19
DEFAULT_FALLBACK_BBOX = [-95.3704, 29.6762, -95.3680, 29.6775]

print(f"Notebook directory: {notebook_dir}")
print(f"Output image: {OUTPUT_IMAGE}")

Notebook directory: /home/megrad/Documents/GitHub/dynamic_search_area/sat_generation
Output image: /home/megrad/Documents/GitHub/dynamic_search_area/sat_generation/satellite.tif


## Step 1: Draw The ROI

Use the search box to jump to a location, then draw the ROI with the polygon tool.

This map is intentionally locked to only two drawing controls:
- polygon, for drawing the ROI, and
- delete, for removing a drawn ROI and starting over.

If you do not draw anything, the notebook will fall back to a small default bbox so the workflow still runs.

In [2]:
m = leafmap.Map(
    center=DEFAULT_CENTER,
    zoom=DEFAULT_MAP_ZOOM,
    draw_control=False,
    search_control=True,
)

polygon_only_draw_control = ipyleaflet.DrawControl(
    polygon={
        "shapeOptions": {"color": "#3388ff", "fillColor": "#3388ff"},
        "repeatMode": False,
    },
    polyline={},
    marker={},
    rectangle={},
    circle={},
    circlemarker={},
    edit=False,
    remove=True,
    position="topleft",
)

m.add(polygon_only_draw_control)
m.draw_control = polygon_only_draw_control

def handle_draw(_, action, geo_json):
    if "style" in geo_json["properties"]:
        del geo_json["properties"]["style"]
    m.user_roi = geo_json

    if action in ["created", "edited"]:
        m.draw_features.append(geo_json)
    elif action == "deleted":
        geometries = [feature["geometry"] for feature in m.draw_control.data]
        for geom in geometries:
            if geom == geo_json["geometry"]:
                geometries.remove(geom)
        for feature in list(m.draw_features):
            if feature["geometry"] not in geometries:
                m.draw_features.remove(feature)

    m.user_rois = {
        "type": "FeatureCollection",
        "features": m.draw_features,
    }

polygon_only_draw_control.on_draw(handle_draw)

m.add_basemap("SATELLITE")
m

Map(center=[29.67684, -95.369222], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title',…

## Step 2: Build The Download Area

This cell does two things:
- captures the original ROI geometry that should remain visible in the final image, and
- expands its bounds into a square bbox for downloading tiles.

That means the raw tile request is square, but the saved GeoTIFF is masked back down to the original shape.

In [6]:
drawn_features = list(m.draw_control.data)

if drawn_features:
    roi_source = "drawn ROI"
    original_geometry = drawn_features[-1]
    original_bbox = leafmap.common.geometry_bounds(original_geometry)
else:
    roi_source = "fallback bbox"
    original_bbox = DEFAULT_FALLBACK_BBOX
    original_geometry = bbox_to_polygon(original_bbox)

download_bbox = square_bbox(original_bbox)

print(f"ROI source: {roi_source}")
print(f"Original bbox: {original_bbox}")
print(f"Square download bbox: {download_bbox}")

ROI source: drawn ROI
Original bbox: [-78.9919, 35.1377, -78.9809, 35.1488]
Square download bbox: [-78.99318720219, 35.1377, -78.97961279781002, 35.148799999999994]


## Step 3: Download The GeoTIFF

Set your intended pixel resolution in meters per pixel, then run the cell.

The notebook will estimate the request size and convert that target resolution to the nearest available web-tile zoom automatically.

In [7]:
TARGET_PIXEL_RESOLUTION = 0.2  # meters per pixel

request_plan = estimate_tms_request(download_bbox, resolution=TARGET_PIXEL_RESOLUTION)
print(f"Target pixel resolution: {TARGET_PIXEL_RESOLUTION} m/px")
print(f"Resolved web-tile zoom: {request_plan['zoom']}")
print(f"Estimated request: {request_plan}")

tms_to_geotiff(
    output=str(OUTPUT_IMAGE),
    bbox=download_bbox,
    resolution=TARGET_PIXEL_RESOLUTION,
    source="Satellite",
    overwrite=True,
    num_workers=8,
    progress_interval=25,
    mask_geometry=original_geometry,
)

print(f"Finished: {OUTPUT_IMAGE}")

Target pixel resolution: 0.2 m/px
Resolved web-tile zoom: 19
Estimated request: {'zoom': 19, 'tile_count': 441, 'xtiles': 21, 'ytiles': 21, 'pixel_width': 5061, 'pixel_height': 5061}
Planned request: {'zoom': 19, 'tile_count': 441, 'xtiles': 21, 'ytiles': 21, 'pixel_width': 5061, 'pixel_height': 5061}
Downloaded tile 1/441
Downloaded tile 25/441
Downloaded tile 50/441
Downloaded tile 75/441
Downloaded tile 100/441
Downloaded tile 125/441
Downloaded tile 150/441
Downloaded tile 175/441
Downloaded tile 200/441
Downloaded tile 225/441
Downloaded tile 250/441
Downloaded tile 275/441
Downloaded tile 300/441
Downloaded tile 325/441
Downloaded tile 350/441
Downloaded tile 375/441
Downloaded tile 400/441
Downloaded tile 425/441
Downloaded tile 441/441
Saving GeoTIFF. Please wait...
Saved GeoTIFF: {'path': '/home/megrad/Documents/GitHub/dynamic_search_area/sat_generation/satellite.tif', 'width': 5061, 'height': 5061, 'crs': 'EPSG:3857'}
Finished: /home/megrad/Documents/GitHub/dynamic_search_are

## Step 4: Inspect The Result

This final cell reports the saved raster dimensions, bounds, and CRS so you can quickly verify the output without opening another tool.

In [8]:
with rasterio.open(OUTPUT_IMAGE) as src:
    print("Path:", OUTPUT_IMAGE)
    print("Size:", src.width, "x", src.height)
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)

Path: /home/megrad/Documents/GitHub/dynamic_search_area/sat_generation/satellite.tif
Size: 5061 x 5061
CRS: EPSG:3857
Bounds: BoundingBox(left=-8793481.375485525, bottom=4182609.8049930213, right=-8791970.279702123, top=4184120.9007764217)
